## Lab: General Linear Regression and Statistical Inference


### Load the libraries.

In [1]:
# CodeGrade step0

from sklearn.datasets import fetch_california_housing
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
import os
import tarfile
import joblib
from sklearn.datasets._base import _pkl_filepath, get_data_home

archive_path = "cal_housing.tgz"
data_home = get_data_home(data_home=None)
if not os.path.exists(data_home):
    os.makedirs(data_home)
filepath = _pkl_filepath(data_home, 'cal_housing.pkz')

with tarfile.open(mode="r:gz", name=archive_path) as f:
    cal_housing = np.loadtxt(
        f.extractfile('CaliforniaHousing/cal_housing.data'),
        delimiter=',')
    columns_index = [8, 7, 2, 3, 4, 5, 6, 1, 0]
    cal_housing = cal_housing[:, columns_index]
    joblib.dump(cal_housing, filepath, compress=6)

california = fetch_california_housing(as_frame=True)
data = california.data
data['MedianHouseValue'] = california.target

In [2]:
# Display basic information
print(data.info())
print(data.describe())

<class 'pandas.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   MedInc            20640 non-null  float64
 1   HouseAge          20640 non-null  float64
 2   AveRooms          20640 non-null  float64
 3   AveBedrms         20640 non-null  float64
 4   Population        20640 non-null  float64
 5   AveOccup          20640 non-null  float64
 6   Latitude          20640 non-null  float64
 7   Longitude         20640 non-null  float64
 8   MedianHouseValue  20640 non-null  float64
dtypes: float64(9)
memory usage: 1.4 MB
None
             MedInc      HouseAge      AveRooms     AveBedrms    Population  \
count  20640.000000  20640.000000  20640.000000  20640.000000  20640.000000   
mean       3.870671     28.639486      5.429000      1.096675   1425.476744   
std        1.899822     12.585558      2.474173      0.473911   1132.462122   
min        0.499900      

### Step 1
Let X be MedInc, AveRooms, and AveOccup with a constant. Let y be MedianHouseValue. Fit mlr_model and return R² (4 decimal places).

In [3]:
# CodeGrade step1

# Define X and y
X = data[['MedInc', 'AveRooms', 'AveOccup']]
y = data['MedianHouseValue']

# Add constant for intercept
X_const = sm.add_constant(X)

# Fit the regression model
mlr_model = sm.OLS(y, X_const).fit()

# Return R² rounded to 4 decimal places
round(mlr_model.rsquared, 4)

np.float64(0.4808)

In [4]:
# Print model summary
print(mlr_model.summary())

                            OLS Regression Results                            
Dep. Variable:       MedianHouseValue   R-squared:                       0.481
Model:                            OLS   Adj. R-squared:                  0.481
Method:                 Least Squares   F-statistic:                     6370.
Date:                Wed, 03 Jun 2026   Prob (F-statistic):               0.00
Time:                        12:15:53   Log-Likelihood:                -25477.
No. Observations:               20640   AIC:                         5.096e+04
Df Residuals:                   20636   BIC:                         5.099e+04
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.6069      0.016     37.444      0.0

### Step 2
Let p_values be the model's p-values. Return the first four p-values using .iloc[], rounded to 5 decimal places.

In [5]:
# CodeGrade step2

# Extract p-values
p_values = mlr_model.pvalues

# Return first four p-values rounded to 5 decimal places
round(p_values.iloc[0], 5), round(p_values.iloc[1], 5), round(p_values.iloc[2], 5), round(p_values.iloc[3], 5)

(np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0))

### Step 3
Identify significant predictors (p < 0.05), calling this significant_predictors. Return the shape.

In [6]:
# CodeGrade step3

# Identify significant predictors (strictly less than alpha=0.05)
significant_predictors = p_values[p_values < 0.05]

# Return shape
significant_predictors.shape

(4,)

### Step 4
Find the 95% confidence intervals, calling this conf_intervals. Return four values using .iloc[,] rounded to 2 decimal places.

In [7]:
# CodeGrade step4

# Compute 95% confidence intervals
conf_intervals = mlr_model.conf_int()

# Return four confidence intervals using .iloc[,], rounded to 2 decimal places
round(conf_intervals.iloc[0, 0], 2), round(conf_intervals.iloc[0, 1], 2), round(conf_intervals.iloc[1, 0], 2), round(conf_intervals.iloc[1, 1], 2)

(np.float64(0.58), np.float64(0.64), np.float64(0.43), np.float64(0.44))

### Step 5
Add MedInc_squared to the model. Fit quad_model and return R² rounded to 4 decimal places.

In [8]:
# CodeGrade step5

# Add quadratic term
data['MedInc_squared'] = data['MedInc'] ** 2

# Define new X with quadratic term
X2 = data[['MedInc', 'AveRooms', 'AveOccup', 'MedInc_squared']]
X2_const = sm.add_constant(X2)

# Fit quadratic model
quad_model = sm.OLS(y, X2_const).fit()

# Return R² rounded to 4 decimal places
round(quad_model.rsquared, 4)

np.float64(0.4858)

In [9]:
# Print quad model summary
print(quad_model.summary())

                            OLS Regression Results                            
Dep. Variable:       MedianHouseValue   R-squared:                       0.486
Model:                            OLS   Adj. R-squared:                  0.486
Method:                 Least Squares   F-statistic:                     4874.
Date:                Wed, 03 Jun 2026   Prob (F-statistic):               0.00
Time:                        12:15:53   Log-Likelihood:                -25378.
No. Observations:               20640   AIC:                         5.077e+04
Df Residuals:                   20635   BIC:                         5.081e+04
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
const              0.3551      0.024     14.

### Step 6
Find adjusted R² for both models. Return both rounded to 4 decimal places, separated by a comma.

In [10]:
# CodeGrade step6

# Adjusted R² for both models
adjusted_r2_base = round(mlr_model.rsquared_adj, 4)
adjusted_r2_quad = round(quad_model.rsquared_adj, 4)

# Return both values
adjusted_r2_base, adjusted_r2_quad

(np.float64(0.4807), np.float64(0.4857))